<a href="https://colab.research.google.com/github/Liza337/MSC-Emo-Sum/blob/main/Ablation_Study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================================================
# ABLATION STUDY: BANGLAT5 WITHOUT EMOTION LABELS
# Upload CSV from desktop in Google Colab
# =========================================================

# Install required packages
# !pip install transformers datasets evaluate bert-score -q
!pip install -q transformers datasets evaluate rouge_score bert-score

# ---------------------------------------------------------
# IMPORT LIBRARIES
# ---------------------------------------------------------
import pandas as pd
import numpy as np
import torch

from google.colab import files
from sklearn.model_selection import train_test_split
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed
)

import evaluate
from bert_score import score as bertscore_score

# ---------------------------------------------------------
# REPRODUCIBILITY
# ---------------------------------------------------------
set_seed(42)

# ---------------------------------------------------------
# UPLOAD CSV FILE FROM DESKTOP
# ---------------------------------------------------------
uploaded = files.upload()

# Get uploaded file name automatically
file_name = list(uploaded.keys())[0]

# Read CSV
df = pd.read_csv(file_name)

print('Dataset loaded successfully!')
print(df.head())
print(df.columns)

# ---------------------------------------------------------
# OPTIONAL: RENAME COLUMNS IF NEEDED
# ---------------------------------------------------------
# Uncomment and edit if your column names are different
# df = df.rename(columns={
#     'Comment': 'comment',
#     'Summary': 'summary',
#     'Emotion': 'emotion'
# })

# ---------------------------------------------------------
# TRAIN-TEST SPLIT (80/20)
# ---------------------------------------------------------
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

print('Train size:', len(train_df))
print('Test size :', len(test_df))

# ---------------------------------------------------------
# ABLATION SETTING: REMOVE EMOTION PREFIX
# ---------------------------------------------------------
train_df = train_df.copy()
test_df  = test_df.copy()

train_df['input_text'] = train_df['comment']
test_df['input_text']  = test_df['comment']

# ---------------------------------------------------------
# MODEL AND TOKENIZER
# ---------------------------------------------------------
MODEL_NAME = 'csebuetnlp/banglat5'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

MAX_INPUT_LEN = 512
MAX_TARGET_LEN = 64

# ---------------------------------------------------------
# TOKENIZATION FUNCTION
# ---------------------------------------------------------
def preprocess_function(batch):

    model_inputs = tokenizer(
        batch['input_text'],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding='max_length'
    )

    labels = tokenizer(
        text_target=batch['summary'],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding='max_length'
    )

    labels_ids = labels['input_ids']
    labels_ids = [
        [(token if token != tokenizer.pad_token_id else -100) for token in seq]
        for seq in labels_ids
    ]

    model_inputs['labels'] = labels_ids
    return model_inputs

# ---------------------------------------------------------
# CREATE DATASETS
# ---------------------------------------------------------
train_ds = Dataset.from_pandas(train_df)
test_ds  = Dataset.from_pandas(test_df)

tokenized_train = train_ds.map(
    preprocess_function,
    batched=True,
    remove_columns=train_df.columns.tolist()
)

tokenized_test = test_ds.map(
    preprocess_function,
    batched=True,
    remove_columns=test_df.columns.tolist()
)

# ---------------------------------------------------------
# LOAD MODEL
# ---------------------------------------------------------
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100
)

# ---------------------------------------------------------
# ROUGE METRICS
# ---------------------------------------------------------
rouge = evaluate.load('rouge')

def compute_metrics(eval_preds):

    preds, labels = eval_preds

    if isinstance(preds, tuple):
        preds = preds[0]

    preds = np.asarray(preds)
    labels = np.asarray(labels)

    preds = np.clip(preds, 0, tokenizer.vocab_size - 1)

    labels_for_decode = np.where(
        labels != -100,
        labels,
        tokenizer.pad_token_id
    )

    decoded_preds = tokenizer.batch_decode(
        preds,
        skip_special_tokens=True
    )

    decoded_labels = tokenizer.batch_decode(
        labels_for_decode,
        skip_special_tokens=True
    )

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )

    result = {k: round(v * 100, 4) for k, v in result.items()}
    return result

# ---------------------------------------------------------
# TRAINING ARGUMENTS
# ---------------------------------------------------------
OUTPUT_DIR = './banglat5_without_emotion'

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=8,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=3e-5,
    weight_decay=0.01,
    logging_steps=200,
    save_total_limit=2,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    fp16=False,
    report_to=[]
)

# ---------------------------------------------------------
# TRAINER
# ---------------------------------------------------------
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# ---------------------------------------------------------
# TRAIN MODEL
# ---------------------------------------------------------
trainer.train()

# ---------------------------------------------------------
# ROUGE EVALUATION
# ---------------------------------------------------------
print('\nRunning ROUGE evaluation on TEST set...')
test_results = trainer.evaluate(eval_dataset=tokenized_test)

print('\n===== BANGLAT5 WITHOUT EMOTION: ROUGE RESULTS =====')
print(f'ROUGE-1 (F1): {test_results["eval_rouge1"]:.4f}%')
print(f'ROUGE-2 (F1): {test_results["eval_rouge2"]:.4f}%')
print(f'ROUGE-L (F1): {test_results["eval_rougeL"]:.4f}%')

# ---------------------------------------------------------
# GENERATE TEST SUMMARIES
# ---------------------------------------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

preds = []
refs = []

batch_size_gen = 4
test_inputs = test_df['input_text'].tolist()
test_refs   = test_df['summary'].tolist()

for i in range(0, len(test_inputs), batch_size_gen):

    batch_inputs = test_inputs[i:i+batch_size_gen]

    enc = tokenizer(
        batch_inputs,
        return_tensors='pt',
        truncation=True,
        padding=True,
        max_length=MAX_INPUT_LEN
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **enc,
            max_new_tokens=MAX_TARGET_LEN,
            num_beams=4
        )

    for out in outputs:
        pred = tokenizer.decode(
            out,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True
        )
        preds.append(pred.strip())

    refs.extend(test_refs[i:i+batch_size_gen])

# ---------------------------------------------------------
# BERTScore
# ---------------------------------------------------------
print('\nComputing BERTScore...')
P, R, F1 = bertscore_score(preds, refs, lang='bn', verbose=True)

print('\n===== BANGLAT5 WITHOUT EMOTION: BERTSCORE =====')
print(f'Precision: {P.mean().item():.4f}')
print(f'Recall   : {R.mean().item():.4f}')
print(f'F1       : {F1.mean().item():.4f}')

# ---------------------------------------------------------
# SAVE PREDICTIONS
# ---------------------------------------------------------
out_df = pd.DataFrame({
    'input': test_inputs,
    'reference': refs,
    'prediction': preds
})

out_csv = 'banglat5_without_emotion_predictions.csv'
out_df.to_csv(out_csv, index=False, encoding='utf-8-sig')

print(f'\nSaved predictions to: {out_csv}')

# ---------------------------------------------------------
# SAVE MODEL
# ---------------------------------------------------------
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print('\nModel saved to:', OUTPUT_DIR)

# ---------------------------------------------------------
# DOWNLOAD PREDICTIONS FILE
# ---------------------------------------------------------
files.download(out_csv)

In [ ]:
import pandas as pd
import evaluate

# Load saved predictions
out_df = pd.read_csv('banglat5_without_emotion_predictions.csv')

# Handle missing/nan values safely
preds = out_df['prediction'].fillna("").astype(str).tolist()
refs = out_df['reference'].fillna("").astype(str).tolist()

# Load rouge
rouge = evaluate.load('rouge')

# Calculate ROUGE without the English stemmer
rouge_results = rouge.compute(
    predictions=preds,
    references=refs,
    use_stemmer=False,
    tokenizer=lambda x: x.split()
)

print('\n===== CORRECTED BANGLAT5 WITHOUT EMOTION: ROUGE RESULTS =====')
print(f'ROUGE-1 (F1): {rouge_results["rouge1"] * 100:.4f}%')
print(f'ROUGE-2 (F1): {rouge_results["rouge2"] * 100:.4f}%')
print(f'ROUGE-L (F1): {rouge_results["rougeL"] * 100:.4f}%')

In [ ]:
!pip install -q sentence-transformers

In [ ]:
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, util

# 1. Load your saved predictions CSV
csv_file = 'banglat5_without_emotion_predictions.csv'
df = pd.read_csv(csv_file)

# 2. Extract inputs, references, and predictions (handling any NaN values)
preds = df['prediction'].fillna("").astype(str).tolist()
refs = df['reference'].fillna("").astype(str).tolist()

print(f"Loaded {len(preds)} predictions from {csv_file}")

# 3. Load the LaBSE Model
print("Loading LaBSE model...")
device = "cuda" if torch.cuda.is_available() else "cpu"
labse_model = SentenceTransformer('sentence-transformers/LaBSE', device=device)

# 4. Generate Embeddings
print("Encoding generated predictions and reference summaries...")
pred_embeddings = labse_model.encode(preds, convert_to_tensor=True, show_progress_bar=True)
ref_embeddings = labse_model.encode(refs, convert_to_tensor=True, show_progress_bar=True)

# 5. Calculate Pairwise Cosine Similarities (row-by-row matching)
# torch.diag extracts the similarity between pred[i] and ref[i]
cosine_scores = torch.nn.functional.cosine_similarity(pred_embeddings, ref_embeddings)
mean_labse_score = cosine_scores.mean().item()

# 6. Display Results
print('\n===== BANGLAT5 WITHOUT EMOTION: LABSE RESULTS =====')
print(f'LaBSE Mean Cosine Similarity: {mean_labse_score:.4f}')

# Optional: Add individual LaBSE scores to your dataframe and resave
df['labse_score'] = cosine_scores.cpu().numpy()
df.to_csv('banglat5_without_emotion_predictions_with_labse.csv', index=False, encoding='utf-8-sig')
print("Updated predictions with LaBSE scores saved to 'banglat5_without_emotion_predictions_with_labse.csv'")